In [2]:
import torch 
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

In [3]:
import torchvision.transforms as transforms
#image-> scale(0,1) -> normalize (-1,1)
transform = transforms.Compose([
    transforms.ToTensor(), # it convert the img into the tensors and scale the img
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) 
                            # Mean     #Std
])
trainset = CIFAR10("./data",train=True,download=True,transform=transform)
testset = CIFAR10("./data",train=False,download=True,transform=transform)

In [4]:
trainloader = DataLoader(trainset,batch_size = 64,shuffle=True)
testloader = DataLoader(testset,batch_size = 64)


### Building CNN

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )
    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) # Flattening
        x = self.fc_layers(x)

        return x
        

In [6]:
model = CNN()

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [8]:
epochs = 10
for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images,labels in trainloader:
        optimizer.zero_grad()

        op = model.forward(images) # FP
        loss = criterion(op,labels) # Loss fnx
        loss.backward() # BP
        optimizer.step() # Update params

        epoch_training_loss+=loss.item()
    print(f"epoch = {epoch} & loss = {epoch_training_loss/len(trainloader)}")

epoch = 0 & loss = 1.3621450490353968
epoch = 1 & loss = 0.9213452065539787
epoch = 2 & loss = 0.7355631029285739
epoch = 3 & loss = 0.6075273198087502
epoch = 4 & loss = 0.49948282583671455
epoch = 5 & loss = 0.4090471927581541
epoch = 6 & loss = 0.3169990529870743
epoch = 7 & loss = 0.24650510899779743
epoch = 8 & loss = 0.1872238943929715
epoch = 9 & loss = 0.14719029626978175


In [12]:
#Evaluation

correct_labels = 0;
tot_labels  = 0;
model.eval();
with torch.no_grad():
    for images,labels  in testloader:
        op = model.forward(images)
        _,pred = torch.max(op,1)
        correct_labels += (pred==labels).sum().item()
        tot_labels  +=labels .size(0)
print(f"accuracy ={(correct_labels /tot_labels)*100}")

accuracy =75.87
